# Raman Spectroscopy: Raw Data to Publication Pipeline\n\n**Sample:** NAM  \n**Instrument:** BWS465-785H (785 nm excitation)  \n**Best Condition:** 70% power, 60s integration, 5 accumulations\n\n### Processing Steps:\n1. Extract Raman Shift + Dark Subtracted\n2. Remove empty/NaN rows\n3. Crop to fingerprint region (400-1800 cm⁻¹)\n4. ALS baseline correction\n5. Savitzky-Golay smoothing\n6. Min-Max normalization\n7. Average replicates\n8. Peak detection\n9. Glass comparison\n10. DFT overlay\n\n---

In [ ]:
# Cell 1: Imports and Configuration\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom openpyxl import load_workbook\nfrom scipy.signal import find_peaks, savgol_filter\nfrom scipy import sparse\nfrom scipy.sparse.linalg import spsolve\nimport os\n\n# Directories\nBASE_DIR = r\"c:\\Users\\sukes\\Downloads\\nam-new\"\nPROCESSED_DIR = os.path.join(BASE_DIR, \"Analysis\", \"Processed\")\nFIGURES_DIR = os.path.join(BASE_DIR, \"Analysis\", \"Figures\")\nos.makedirs(PROCESSED_DIR, exist_ok=True)\nos.makedirs(FIGURES_DIR, exist_ok=True)\nos.chdir(BASE_DIR)\n\n# Plot settings\nplt.rcParams.update({\n    'font.size': 11,\n    'axes.linewidth': 1.2,\n    'xtick.major.width': 1.2,\n    'ytick.major.width': 1.2,\n    'figure.dpi': 150,\n    'savefig.dpi': 300,\n    'savefig.bbox': 'tight'\n})\n\nprint(\"Setup complete.\")\nprint(f\"Base: {BASE_DIR}\")\nprint(f\"Processed: {PROCESSED_DIR}\")\nprint(f\"Figures: {FIGURES_DIR}\")

## Step 1: Extract Raman Shift + Dark Subtracted\n\nFrom each XLSX: Column 4 (Raman Shift) + Column 8 (Dark Subtracted #1)

In [ ]:
# Step 1: Extract spectrum from XLSX\n\ndef extract_spectrum(xlsx_path):\n    \"\"\"Extract Raman Shift and Dark Subtracted intensity from XLSX.\"\"\"\n    wb = load_workbook(xlsx_path, read_only=True)\n    ws = wb.active\n    raman_shift = []\n    intensity = []\n    for row in ws.iter_rows(min_row=100, values_only=True):\n        try:\n            rs = float(row[3])   # Column 4: Raman Shift\n            ds = float(row[7])   # Column 8: Dark Subtracted #1\n            raman_shift.append(rs)\n            intensity.append(ds)\n        except (TypeError, ValueError, IndexError):\n            continue\n    wb.close()\n    return np.array(raman_shift), np.array(intensity)\n\n# Load NAM spectra (best condition: 70%, 60s, 5acc)\nnam_dir = r\"70p-60s-5ac-diifrentpoint\"\nnam_files = sorted([f for f in os.listdir(nam_dir) if f.endswith('.xlsx')])\n\nnam_raw = {}\nfor f in nam_files:\n    rs, intensity = extract_spectrum(os.path.join(nam_dir, f))\n    nam_raw[f] = {'raman_shift': rs, 'intensity': intensity}\n    print(f\"  {f}: {len(rs)} points, range {rs.min():.1f} to {rs.max():.1f} cm-1\")\n\n# Load Glass spectra\nglass_dir = r\"glass slide empty\\empty slide\"\nglass_files = sorted([f for f in os.listdir(glass_dir) if f.endswith('.xlsx')])\n\nglass_raw = {}\nfor f in glass_files:\n    rs, intensity = extract_spectrum(os.path.join(glass_dir, f))\n    glass_raw[f] = {'raman_shift': rs, 'intensity': intensity}\n\nprint(f\"\\nNAM: {len(nam_raw)} spectra loaded\")\nprint(f\"Glass: {len(glass_raw)} spectra loaded\")

## Step 2: Remove Empty Rows + Step 3: Crop Region\n\n- Remove NaN/blank/non-numeric rows\n- Fingerprint region: **400-1800 cm⁻¹**\n- Full spectrum (supplementary): **400-3500 cm⁻¹**

In [ ]:
# Steps 2 & 3: Clean data and crop to region\n\ndef clean_and_crop(raman_shift, intensity, region=(400, 1800)):\n    \"\"\"Remove NaN, non-numeric, and crop to spectral region.\"\"\"\n    # Remove NaN\n    valid = ~(np.isnan(raman_shift) | np.isnan(intensity))\n    rs = raman_shift[valid]\n    inten = intensity[valid]\n    \n    # Crop to region\n    mask = (rs >= region[0]) & (rs <= region[1])\n    return rs[mask], inten[mask]\n\n# Process NAM - fingerprint region\nnam_fingerprint = {}\nfor f, data in nam_raw.items():\n    rs, inten = clean_and_crop(data['raman_shift'], data['intensity'], region=(400, 1800))\n    nam_fingerprint[f] = {'raman_shift': rs, 'intensity': inten}\n\n# Process NAM - full spectrum\nnam_full = {}\nfor f, data in nam_raw.items():\n    rs, inten = clean_and_crop(data['raman_shift'], data['intensity'], region=(400, 2800))\n    nam_full[f] = {'raman_shift': rs, 'intensity': inten}\n\n# Process Glass - fingerprint region\nglass_fingerprint = {}\nfor f, data in glass_raw.items():\n    rs, inten = clean_and_crop(data['raman_shift'], data['intensity'], region=(400, 1800))\n    glass_fingerprint[f] = {'raman_shift': rs, 'intensity': inten}\n\n# Verify\nfor f, data in nam_fingerprint.items():\n    print(f\"  {f}: {len(data['raman_shift'])} points ({data['raman_shift'].min():.0f}-{data['raman_shift'].max():.0f} cm-1)\")\n\nprint(f\"\\nFingerprint region: 400-1800 cm-1\")\nprint(f\"Full spectrum: 400-2800 cm-1\")

## Step 4: ALS Baseline Correction\n\n**Asymmetric Least Squares (ALS)**  \n- lambda = 1e5 (smoothness)\n- p = 0.001 (asymmetry)\n- iterations = 10

In [ ]:
# Step 4: ALS Baseline Correction\n\ndef baseline_als(y, lam=1e5, p=0.001, niter=10):\n    \"\"\"Asymmetric Least Squares baseline correction.\n    \n    Parameters:\n        y: signal\n        lam: smoothness (larger = smoother baseline)\n        p: asymmetry (smaller = more asymmetric, hugs bottom)\n        niter: number of iterations\n    \"\"\"\n    L = len(y)\n    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))\n    w = np.ones(L)\n    \n    for i in range(niter):\n        W = sparse.spdiags(w, 0, L, L)\n        Z = W + lam * D.dot(D.T)\n        z = spsolve(Z, w * y)\n        w = p * (y > z) + (1 - p) * (y < z)\n    \n    return z\n\n# Apply baseline correction to NAM fingerprint spectra\nnam_baseline_corrected = {}\nfor f, data in nam_fingerprint.items():\n    baseline = baseline_als(data['intensity'], lam=1e5, p=0.001, niter=10)\n    corrected = data['intensity'] - baseline\n    nam_baseline_corrected[f] = {\n        'raman_shift': data['raman_shift'],\n        'intensity': corrected,\n        'baseline': baseline,\n        'raw': data['intensity']\n    }\n\n# Apply to Glass\nglass_baseline_corrected = {}\nfor f, data in glass_fingerprint.items():\n    baseline = baseline_als(data['intensity'], lam=1e5, p=0.001, niter=10)\n    corrected = data['intensity'] - baseline\n    glass_baseline_corrected[f] = {\n        'raman_shift': data['raman_shift'],\n        'intensity': corrected,\n        'baseline': baseline,\n        'raw': data['intensity']\n    }\n\nprint(\"Baseline correction applied to all spectra.\")\nprint(f\"  NAM: {len(nam_baseline_corrected)} spectra\")\nprint(f\"  Glass: {len(glass_baseline_corrected)} spectra\")

In [ ]:
# Visualize baseline correction on one spectrum\nf = list(nam_baseline_corrected.keys())[0]\ndata = nam_baseline_corrected[f]\n\nfig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)\nfig.subplots_adjust(hspace=0.1)\n\n# Top: Raw + Baseline\nax1.plot(data['raman_shift'], data['raw'], 'b-', lw=0.7, label='Raw')\nax1.plot(data['raman_shift'], data['baseline'], 'r--', lw=1.5, label='ALS Baseline')\nax1.set_ylabel('Intensity (a.u.)')\nax1.set_title(f'Baseline Correction: {f}', fontweight='bold', loc='left')\nax1.legend(loc='upper right')\n\n# Bottom: Corrected\nax2.plot(data['raman_shift'], data['intensity'], 'k-', lw=0.7, label='Corrected')\nax2.axhline(0, color='gray', lw=0.5, ls='--')\nax2.set_xlabel('Raman Shift (cm$^{-1}$)')\nax2.set_ylabel('Intensity (a.u.)')\nax2.set_title('After Baseline Subtraction', fontweight='bold', loc='left')\nax2.legend(loc='upper right')\n\nplt.tight_layout()\nplt.show()\nprint(\"Baseline is flat, peaks preserved.\")

## Step 5: Savitzky-Golay Smoothing\n\n- Window length = 11\n- Polynomial order = 3\n- Reduces noise while preserving peak shape

In [ ]:
# Step 5: Savitzky-Golay Smoothing\n\nSG_WINDOW = 11\nSG_POLY = 3\n\n# Apply to NAM\nnam_smoothed = {}\nfor f, data in nam_baseline_corrected.items():\n    smoothed = savgol_filter(data['intensity'], window_length=SG_WINDOW, polyorder=SG_POLY)\n    nam_smoothed[f] = {\n        'raman_shift': data['raman_shift'],\n        'intensity': smoothed,\n        'unsmoothed': data['intensity']\n    }\n\n# Apply to Glass\nglass_smoothed = {}\nfor f, data in glass_baseline_corrected.items():\n    smoothed = savgol_filter(data['intensity'], window_length=SG_WINDOW, polyorder=SG_POLY)\n    glass_smoothed[f] = {\n        'raman_shift': data['raman_shift'],\n        'intensity': smoothed\n    }\n\nprint(f\"Savitzky-Golay applied: window={SG_WINDOW}, poly={SG_POLY}\")\nprint(f\"  NAM: {len(nam_smoothed)} spectra smoothed\")\nprint(f\"  Glass: {len(glass_smoothed)} spectra smoothed\")

## Step 6: Min-Max Normalization\n\nI_norm = (I - I_min) / (I_max - I_min) → range [0, 1]

In [ ]:
# Step 6: Min-Max Normalization\n\ndef normalize_minmax(intensity):\n    \"\"\"Normalize to [0, 1] range.\"\"\"\n    imin = intensity.min()\n    imax = intensity.max()\n    if imax - imin == 0:\n        return np.zeros_like(intensity)\n    return (intensity - imin) / (imax - imin)\n\n# Apply to NAM\nnam_normalized = {}\nfor f, data in nam_smoothed.items():\n    norm = normalize_minmax(data['intensity'])\n    nam_normalized[f] = {\n        'raman_shift': data['raman_shift'],\n        'intensity': norm\n    }\n\n# Apply to Glass\nglass_normalized = {}\nfor f, data in glass_smoothed.items():\n    norm = normalize_minmax(data['intensity'])\n    glass_normalized[f] = {\n        'raman_shift': data['raman_shift'],\n        'intensity': norm\n    }\n\nprint(\"Min-Max normalization applied.\")\nfor f, data in nam_normalized.items():\n    print(f\"  {f}: min={data['intensity'].min():.3f}, max={data['intensity'].max():.3f}\")

## Step 7: Average Replicates\n\nCompute mean and standard deviation across the 3 NAM spots.

In [ ]:
# Step 7: Average Replicates\n\n# Use common raman_shift (all should be identical)\nrs_common = list(nam_normalized.values())[0]['raman_shift']\n\n# Stack all NAM intensities\nnam_stack = np.array([data['intensity'] for data in nam_normalized.values()])\n\n# Compute mean and std\nnam_mean = nam_stack.mean(axis=0)\nnam_std = nam_stack.std(axis=0)\n\nprint(f\"Averaged {nam_stack.shape[0]} NAM spectra\")\nprint(f\"Mean range: {nam_mean.min():.4f} to {nam_mean.max():.4f}\")\nprint(f\"Max std: {nam_std.max():.4f}\")\n\n# Same for Glass\nrs_glass = list(glass_normalized.values())[0]['raman_shift']\nglass_stack = np.array([data['intensity'] for data in glass_normalized.values()])\nglass_mean = glass_stack.mean(axis=0)\nglass_std = glass_stack.std(axis=0)\n\nprint(f\"\\nAveraged {glass_stack.shape[0]} Glass spectra\")

## Step 8: Peak Detection\n\nDetect peaks in the averaged NAM spectrum.

In [ ]:
# Step 8: Peak Detection\n\npeaks, properties = find_peaks(nam_mean, prominence=0.02, distance=10)\n\npeak_positions = rs_common[peaks]\npeak_intensities = nam_mean[peaks]\npeak_prominences = properties['prominences']\n\nprint(f\"Peaks detected: {len(peaks)}\")\nprint(f\"\\n{'Peak #':<8} {'Position (cm-1)':<18} {'Intensity':<12} {'Prominence':<12}\")\nprint(\"-\" * 50)\nfor i, (pos, inten, prom) in enumerate(zip(peak_positions, peak_intensities, peak_prominences), 1):\n    print(f\"{i:<8} {pos:<18.1f} {inten:<12.4f} {prom:<12.4f}\")

## Step 9: Glass vs NAM Comparison (Figure 1)\n\nOverlay processed Glass and NAM spectra to confirm glass does not contribute peaks.

In [ ]:
# Step 9: Figure 1 — Glass vs NAM (Processed)\n\nfig, ax = plt.subplots(figsize=(10, 5))\n\n# Glass (mean + std shading)\nax.fill_between(rs_glass, glass_mean - glass_std, glass_mean + glass_std,\n                alpha=0.3, color='blue', label='Glass (std)')\nax.plot(rs_glass, glass_mean, 'b-', lw=1.0, label='Glass (mean)')\n\n# NAM (mean + std shading)\nax.fill_between(rs_common, nam_mean - nam_std, nam_mean + nam_std,\n                alpha=0.3, color='red', label='NAM (std)')\nax.plot(rs_common, nam_mean, 'r-', lw=1.0, label='NAM (mean)')\n\n# Mark NAM peaks\nax.plot(peak_positions, peak_intensities, 'kv', markersize=5)\nfor pos, inten in zip(peak_positions, peak_intensities):\n    if inten > 0.1:  # Only label significant peaks\n        ax.annotate(f'{pos:.0f}', xy=(pos, inten), xytext=(0, 8),\n                    textcoords='offset points', ha='center', fontsize=7, color='darkred')\n\nax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12)\nax.set_ylabel('Normalized Intensity', fontsize=12)\nax.set_title('Figure 1: Glass Substrate vs NAM (Processed)', fontsize=13, fontweight='bold')\nax.legend(loc='upper right', fontsize=10)\nax.set_xlim(400, 1800)\nax.set_ylim(-0.05, 1.1)\n\nplt.tight_layout()\nplt.savefig(os.path.join(FIGURES_DIR, 'Figure1_Glass_vs_NAM.png'), dpi=300, facecolor='white')\nplt.savefig(os.path.join(FIGURES_DIR, 'Figure1_Glass_vs_NAM.pdf'), facecolor='white')\nplt.show()\nprint(\"Figure 1 saved.\")

## Step 10: DFT Preparation (Placeholder)\n\nWhen Gaussian output is available:\n- Extract Frequency + Raman Activity\n- Apply scaling factor: Scaled Frequency = Frequency x 0.96\n- Plot as stick spectrum below experimental

In [ ]:
# Step 10 & 11: DFT Overlay (placeholder — update when Gaussian output is ready)\n\ndef load_dft_spectrum(dft_file):\n    \"\"\"Load DFT Raman spectrum from Gaussian output.\n    \n    Expected format: two columns - Frequency, Raman Activity\n    Apply scaling factor of 0.96 to frequency.\n    \"\"\"\n    data = np.loadtxt(dft_file, delimiter=',')\n    freq_scaled = data[:, 0] * 0.96  # Scaling factor\n    activity = data[:, 1]\n    return freq_scaled, activity\n\ndef broaden_dft(freq, activity, x_range, fwhm=8):\n    \"\"\"Broaden stick spectrum with Lorentzian for comparison.\"\"\"\n    x = np.linspace(x_range[0], x_range[1], 2000)\n    y = np.zeros_like(x)\n    gamma = fwhm / 2\n    for f, a in zip(freq, activity):\n        y += a * (gamma**2) / ((x - f)**2 + gamma**2)\n    # Normalize\n    if y.max() > 0:\n        y = y / y.max()\n    return x, y\n\nprint(\"DFT functions ready.\")\nprint(\"To use: place your Gaussian frequency file in Analysis/DFT/\")\nprint(\"Then run: freq, activity = load_dft_spectrum('Analysis/DFT/your_file.csv')\")

## Figure 4: Final Publication Raman Spectrum\n\nThe main figure showing the fully processed NAM spectrum with peak assignments.

In [ ]:
# Figure 4: Final Publication-Quality Raman Spectrum\n\nfig, ax = plt.subplots(figsize=(10, 4.5))\n\n# Mean spectrum with std shading\nax.fill_between(rs_common, nam_mean - nam_std, nam_mean + nam_std,\n                alpha=0.2, color='black')\nax.plot(rs_common, nam_mean, 'k-', lw=1.0)\n\n# Peak markers and labels\nfor pos, inten, prom in zip(peak_positions, peak_intensities, peak_prominences):\n    if prom > 0.03:  # Only label prominent peaks\n        ax.annotate(f'{pos:.0f}',\n                    xy=(pos, inten), xytext=(0, 8),\n                    textcoords='offset points', ha='center',\n                    fontsize=8, fontweight='bold', color='red')\n        ax.plot(pos, inten, 'rv', markersize=4)\n\nax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12, fontweight='bold')\nax.set_ylabel('Normalized Intensity (a.u.)', fontsize=12, fontweight='bold')\nax.set_xlim(400, 1800)\nax.set_ylim(-0.02, 1.05)\nax.tick_params(axis='both', labelsize=10)\n\n# Remove top and right spines for publication style\nax.spines['top'].set_visible(False)\nax.spines['right'].set_visible(False)\n\nplt.tight_layout()\nplt.savefig(os.path.join(FIGURES_DIR, 'Figure4_Final_Raman.png'), dpi=300, facecolor='white')\nplt.savefig(os.path.join(FIGURES_DIR, 'Figure4_Final_Raman.pdf'), facecolor='white')\nplt.show()\nprint(\"Figure 4 (Final Raman) saved.\")

## Save Processed Data to XLSX

In [ ]:
# Save all processed data to Processed/ folder\n\n# Save individual NAM processed spectra\nfor i, (f, data) in enumerate(nam_normalized.items(), 1):\n    df = pd.DataFrame({\n        'Raman Shift (cm-1)': data['raman_shift'],\n        'Normalized Intensity': data['intensity']\n    })\n    out_name = f\"NAM_70P_60S_5AC_spot{i}_processed.xlsx\"\n    df.to_excel(os.path.join(PROCESSED_DIR, out_name), index=False)\n    print(f\"  Saved: {out_name}\")\n\n# Save NAM average\ndf_avg = pd.DataFrame({\n    'Raman Shift (cm-1)': rs_common,\n    'Mean Intensity': nam_mean,\n    'Std': nam_std\n})\ndf_avg.to_excel(os.path.join(PROCESSED_DIR, 'NAM_average.xlsx'), index=False)\nprint(f\"  Saved: NAM_average.xlsx\")\n\n# Save Glass processed\ndf_glass = pd.DataFrame({\n    'Raman Shift (cm-1)': rs_glass,\n    'Mean Intensity': glass_mean,\n    'Std': glass_std\n})\ndf_glass.to_excel(os.path.join(PROCESSED_DIR, 'glass_processed.xlsx'), index=False)\nprint(f\"  Saved: glass_processed.xlsx\")\n\n# Save peak list\ndf_peaks = pd.DataFrame({\n    'Peak Position (cm-1)': peak_positions,\n    'Intensity': peak_intensities,\n    'Prominence': peak_prominences\n})\ndf_peaks.to_excel(os.path.join(PROCESSED_DIR, 'NAM_peak_list.xlsx'), index=False)\nprint(f\"  Saved: NAM_peak_list.xlsx\")\n\nprint(f\"\\nAll processed files saved to: {PROCESSED_DIR}\")

## Processing Summary\n\nVisualize the full processing pipeline on one spectrum.

In [ ]:
# Processing pipeline summary — show all steps on one spectrum\n\nf0 = list(nam_raw.keys())[0]\n\nfig, axes = plt.subplots(5, 1, figsize=(10, 12), sharex=True)\nfig.subplots_adjust(hspace=0.15)\n\n# (a) Raw\nrs_raw = nam_raw[f0]['raman_shift']\nint_raw = nam_raw[f0]['intensity']\nmask_raw = (rs_raw >= 400) & (rs_raw <= 1800)\naxes[0].plot(rs_raw[mask_raw], int_raw[mask_raw], 'b-', lw=0.6)\naxes[0].set_title('(a) Raw Spectrum', fontweight='bold', loc='left')\naxes[0].set_ylabel('Counts')\n\n# (b) After baseline correction\ndata_bc = nam_baseline_corrected[f0]\naxes[1].plot(data_bc['raman_shift'], data_bc['raw'], 'b-', lw=0.5, alpha=0.5, label='Raw')\naxes[1].plot(data_bc['raman_shift'], data_bc['baseline'], 'r-', lw=1.5, label='Baseline')\naxes[1].set_title('(b) ALS Baseline Fitting', fontweight='bold', loc='left')\naxes[1].set_ylabel('Counts')\naxes[1].legend(loc='upper right', fontsize=8)\n\n# (c) Baseline corrected\naxes[2].plot(data_bc['raman_shift'], data_bc['intensity'], 'g-', lw=0.6)\naxes[2].axhline(0, color='gray', lw=0.5, ls='--')\naxes[2].set_title('(c) Baseline Corrected', fontweight='bold', loc='left')\naxes[2].set_ylabel('Intensity')\n\n# (d) Smoothed\ndata_sm = nam_smoothed[f0]\naxes[3].plot(data_sm['raman_shift'], data_sm['unsmoothed'], 'gray', lw=0.4, alpha=0.5, label='Unsmoothed')\naxes[3].plot(data_sm['raman_shift'], data_sm['intensity'], 'k-', lw=0.8, label='SG Smoothed')\naxes[3].set_title('(d) Savitzky-Golay Smoothing', fontweight='bold', loc='left')\naxes[3].set_ylabel('Intensity')\naxes[3].legend(loc='upper right', fontsize=8)\n\n# (e) Normalized\ndata_norm = nam_normalized[f0]\naxes[4].plot(data_norm['raman_shift'], data_norm['intensity'], 'k-', lw=0.8)\naxes[4].set_title('(e) Min-Max Normalized', fontweight='bold', loc='left')\naxes[4].set_xlabel('Raman Shift (cm$^{-1}$)')\naxes[4].set_ylabel('Norm. Int.')\naxes[4].set_xlim(400, 1800)\n\nplt.savefig(os.path.join(FIGURES_DIR, 'Figure_Processing_Pipeline.png'), dpi=300, facecolor='white')\nplt.savefig(os.path.join(FIGURES_DIR, 'Figure_Processing_Pipeline.pdf'), facecolor='white')\nplt.show()\nprint(\"Processing pipeline figure saved.\")